In [1]:
import pyzx as zx
import os
import json
from collections import Counter

In [91]:
seed = 227
q, d = 10, 300
sigma = None
strSigma = "Inf" if sigma==None else sigma

g = zx.generate.cliffordT(qubits=q, depth=d, seed=seed, sigma=sigma)
g = zx.simplify.teleport_reduce(g) ##*****
c = zx.Circuit.from_graph(g)
zx.draw(c,scale=20)
qasm = c.to_qasm()

zx.tcount(c)

39

In [92]:
gate_count = len(c.gates)

gate_counts = Counter(type(g).__name__ for g in c.gates)
print(gate_counts)

#stats = c.stats_dict()
#print(stats)

circ_depth = c.depth()

Counter({'S': 106, 'CNOT': 69, 'XPhase': 65, 'T': 39})


In [93]:
tcount_raw = zx.tcount(g)
zx.simplify.full_reduce(g)
tcount_simp = zx.tcount(g)

KeyError: 13

In [94]:
g = zx.qasm(qasm)
g = g.to_graph()
zx.draw(g,scale=20)
g.apply_state("0"*q)
g.apply_effect("0"*q)
#zx.draw(g,scale=20)
zx.simplify.full_reduce(g)
tcount_plug_simp = zx.tcount(g)
g.normalize()
zx.draw(g,scale=20)
g.scalar.to_number()

(-4194304.000000014-4194304.000000012j)

In [95]:
print(tcount_raw)
#print(tcount_simp)
print(tcount_plug_simp)

39
20


In [96]:
%%time
amplitude = zx.simulation.simulate(zx.simulation.Strategy.MAGIC_CAT, g)
print(amplitude)

(-0.0013404130879203814+0.04715260864009945j)
CPU times: total: 109 ms
Wall time: 104 ms


In [8]:
path = "circuits/random_clifford_t"
name = f"rct_q{q}_d{circ_depth}_sig{strSigma}_s{seed}"
os.makedirs(path, exist_ok=True)

# save qasm file
filename = os.path.join(path, name+".qasm")
with open(filename, "w") as f:
    f.write(qasm)

# save json file
data = {
    "name": name,
    "file": f"{name}.qasm",
    "provenance": {
        "generator": "pyzx.generate.cliffordT",
        "version": zx.__version__,
        "parameters": {
            "qubits": q,
            "depth": d,
            "seed": seed,
            "sigma": str(sigma)
        },
        "description": f"pyzx[{zx.__version__}].generate.cliffordT(qubits={q}, depth={d}, seed={seed}, sigma={sigma})"
    },
    "metrics": {
        "qubits": q,
        "depth": circ_depth,
        "sigma": strSigma,
        "t_count": tcount_raw,
        "gate_count": gate_count,
        "cnot_count": gate_counts["CNOT"],
        "had_count": gate_counts["HAD"]
    },
    "derived_metrics": {
        "pyzx": {
            "version": zx.__version__,
            "full_reduce_t_count": tcount_simp,
            "plugged_full_reduce_t_count": tcount_plug_simp
        }
    },
    "amplitude": {
        "real": amplitude.real,
        "imag": amplitude.imag
    }
}
filename = os.path.join(path, name+".json")
with open(filename, "w") as f:
    json.dump(data, f, indent=4)

# Generate Dataset

In [2]:
def gen_clifford_t(q,d,sigma,seed,comp_amp=True):
    # generate circuit
    g = zx.generate.cliffordT(qubits=q, depth=d, seed=seed, sigma=sigma)
    g = zx.simplify.teleport_reduce(g) #***
    c = zx.Circuit.from_graph(g)
    #zx.draw(c,scale=20)
    qasm = c.to_qasm()

    # measure raw metrics
    strSigma = "Inf" if sigma==None else sigma
    gate_count = len(c.gates)
    gate_counts = Counter(type(g).__name__ for g in c.gates)
    circ_depth = c.depth()

    # measure (unplugged) pyzx metrics
    tcount_raw = zx.tcount(g)
    #zx.simplify.full_reduce(g)
    #tcount_simp = zx.tcount(g)

    # measure plugged pyzx metrics (and prep for pyzx simulation)
    g = zx.qasm(qasm)
    g = g.to_graph()
    #zx.draw(g,scale=20)
    g.apply_state("0"*q)
    g.apply_effect("0"*q)
    zx.simplify.full_reduce(g)
    tcount_plug_simp = zx.tcount(g)
    g.normalize()
    #zx.draw(g,scale=20)
    #print(g.scalar.to_number())

    # simulate (to measure 'ground truth' amplitude)
    if comp_amp: amplitude = zx.simulation.simulate(zx.simulation.Strategy.MAGIC_CAT, g) #TEMP*

    # =============
    # PRINT TO JSON
    # =============

    # make file
    path = "circuits/random_clifford_t"
    name = f"rct_q{q}_d{circ_depth}_sig{strSigma}_s{seed}"
    if (tcount_plug_simp > 50): return False #TEMP
    print("t",tcount_plug_simp,"\t",name) #TEMP
    os.makedirs(path, exist_ok=True)

    # save qasm file
    filename = os.path.join(path, name+".qasm")
    with open(filename, "w") as f:
        f.write(qasm)

    # save json file
    data = {
        "display_name": name,
        "file": f"{name}.qasm",
        "circuit_class": "random_clifford_t",
        "benchmark_set": "random_clifford_t",
        "provenance": {
            "generator": "pyzx.generate.cliffordT",
            "version": zx.__version__,
            "parameters": {
                "qubits": q,
                "depth": d,
                "seed": seed,
                "sigma": str(sigma)
            },
            "description": f"pyzx[{zx.__version__}].generate.cliffordT(qubits={q}, depth={d}, seed={seed}, sigma={sigma})"
        },
        "metrics": {
            "qubits": q,
            "depth": circ_depth,
            "sigma": strSigma,
            "t_count": tcount_raw,
            "pyzx_reduced_t_count": tcount_plug_simp,
            "gate_count": gate_count,
            "cnot_count": gate_counts["CNOT"],
            "had_count": gate_counts["HAD"]
        },
        "amplitude": {
            "real": amplitude.real if comp_amp else "TBC",
            "imag": amplitude.imag if comp_amp else "TBC"
        }
    }
    filename = os.path.join(path, name+".json")
    with open(filename, "w") as f:
        json.dump(data, f, indent=4)
    
    return True

In [3]:
gen_clifford_t(10,250,None,1,True)
gen_clifford_t(10,250,None,2,True)
gen_clifford_t(10,250,None,3,True)
gen_clifford_t(10,500,None,3,True)

t 18 	 rct_q10_d50_sigInf_s1
t 15 	 rct_q10_d65_sigInf_s2
t 21 	 rct_q10_d56_sigInf_s3
t 42 	 rct_q10_d107_sigInf_s3


True

In [3]:
%%time
i = 0
for q in range(10,60,10):
    for d in range(20,120,20):
        for s in [0,1,2,3,5,10,None]:
            i+=1
            print("             > ",i)
            gen_clifford_t(q,d*10,s,1,False)

             >  1
t 6 	 rct_q10_d39_sig0_s1
             >  2
t 0 	 rct_q10_d48_sig1_s1
             >  3
t 15 	 rct_q10_d45_sig2_s1
             >  4
t 13 	 rct_q10_d50_sig3_s1
             >  5
t 12 	 rct_q10_d50_sig5_s1
             >  6
t 13 	 rct_q10_d50_sig10_s1
             >  7
t 7 	 rct_q10_d42_sigInf_s1
             >  8
t 23 	 rct_q10_d77_sig0_s1
             >  9
t 25 	 rct_q10_d80_sig1_s1
             >  10
t 37 	 rct_q10_d84_sig2_s1
             >  11
t 45 	 rct_q10_d93_sig3_s1
             >  12
t 43 	 rct_q10_d90_sig5_s1
             >  13
t 47 	 rct_q10_d87_sig10_s1
             >  14
t 36 	 rct_q10_d80_sigInf_s1
             >  15
t 41 	 rct_q10_d110_sig0_s1
             >  16
t 48 	 rct_q10_d119_sig1_s1
             >  17
             >  18
             >  19
             >  20
             >  21
             >  22
             >  23
             >  24
             >  25
             >  26
             >  27
             >  28
             >  29
             >  30
  

In [ ]:
# *Maybe instead of computing the ground truth amplitude here, we'll just run the benchmarks with the various simulation methods on the VM
# and take the amplitudes (either aggregate or pick one method) from there (and add it to the circuit json files)x

In [4]:
def gen_fixed_q(q,d,sigma,seed,comp_amp=True):
    # generate circuit
    g = zx.generate.cliffordT(qubits=q, depth=d, seed=seed, sigma=sigma)
    g = zx.simplify.teleport_reduce(g) #***
    c = zx.Circuit.from_graph(g)
    #zx.draw(c,scale=20)
    qasm = c.to_qasm()

    # measure raw metrics
    strSigma = "Inf" if sigma==None else sigma
    gate_count = len(c.gates)
    gate_counts = Counter(type(g).__name__ for g in c.gates)
    circ_depth = c.depth()

    # measure (unplugged) pyzx metrics
    tcount_raw = zx.tcount(g)
    #zx.simplify.full_reduce(g)
    #tcount_simp = zx.tcount(g)

    # measure plugged pyzx metrics (and prep for pyzx simulation)
    g = zx.qasm(qasm)
    g = g.to_graph()
    #zx.draw(g,scale=20)
    g.apply_state("0"*q)
    g.apply_effect("0"*q)
    zx.simplify.full_reduce(g)
    tcount_plug_simp = zx.tcount(g)
    g.normalize()
    #zx.draw(g,scale=20)
    #print(g.scalar.to_number())

    # simulate (to measure 'ground truth' amplitude)
    if comp_amp: amplitude = zx.simulation.simulate(zx.simulation.Strategy.MAGIC_CAT, g) #TEMP*

    # =============
    # PRINT TO JSON
    # =============

    # make file
    class_name = "random_fixed_q20"
    path = "circuits/"+class_name
    name = f"rfq_q{q}_d{circ_depth}_sig{strSigma}_s{seed}"
    if (tcount_plug_simp > 50 or tcount_plug_simp < 1): return False #TEMP
    print("t",tcount_plug_simp,"\t",name) #TEMP
    os.makedirs(path, exist_ok=True)

    # save qasm file
    filename = os.path.join(path, name+".qasm")
    with open(filename, "w") as f:
        f.write(qasm)

    # save json file
    data = {
        "display_name": name,
        "file": f"{name}.qasm",
        "circuit_class": class_name,
        "benchmark_set": class_name,
        "provenance": {
            "generator": "pyzx.generate.cliffordT",
            "version": zx.__version__,
            "parameters": {
                "qubits": q,
                "depth": d,
                "seed": seed,
                "sigma": str(sigma)
            },
            "description": f"pyzx[{zx.__version__}].generate.cliffordT(qubits={q}, depth={d}, seed={seed}, sigma={sigma})"
        },
        "metrics": {
            "qubits": q,
            "depth": circ_depth,
            "sigma": strSigma,
            "t_count": tcount_raw,
            "pyzx_reduced_t_count": tcount_plug_simp,
            "gate_count": gate_count,
            "cnot_count": gate_counts["CNOT"],
            "had_count": gate_counts["HAD"]
        },
        "amplitude": {
            "real": amplitude.real if comp_amp else "TBC",
            "imag": amplitude.imag if comp_amp else "TBC"
        }
    }
    filename = os.path.join(path, name+".json")
    with open(filename, "w") as f:
        json.dump(data, f, indent=4)
    
    return True


def gen_fixed_t(q,d,sigma,seed,comp_amp=True):
    # generate circuit
    g = zx.generate.cliffordT(qubits=q, depth=d, seed=seed, sigma=sigma)
    g = zx.simplify.teleport_reduce(g) #***
    c = zx.Circuit.from_graph(g)
    #zx.draw(c,scale=20)
    qasm = c.to_qasm()

    # measure raw metrics
    strSigma = "Inf" if sigma==None else sigma
    gate_count = len(c.gates)
    gate_counts = Counter(type(g).__name__ for g in c.gates)
    circ_depth = c.depth()

    # measure (unplugged) pyzx metrics
    tcount_raw = zx.tcount(g)
    #zx.simplify.full_reduce(g)
    #tcount_simp = zx.tcount(g)

    # measure plugged pyzx metrics (and prep for pyzx simulation)
    g = zx.qasm(qasm)
    g = g.to_graph()
    #zx.draw(g,scale=20)
    g.apply_state("0"*q)
    g.apply_effect("0"*q)
    zx.simplify.full_reduce(g)
    tcount_plug_simp = zx.tcount(g)
    g.normalize()
    #zx.draw(g,scale=20)
    #print(g.scalar.to_number())

    # simulate (to measure 'ground truth' amplitude)
    if comp_amp: amplitude = zx.simulation.simulate(zx.simulation.Strategy.MAGIC_CAT, g) #TEMP*

    # =============
    # PRINT TO JSON
    # =============

    # make file
    class_name = "random_fixed_t30"
    path = "circuits/"+class_name
    name = f"rct_q{q}_d{circ_depth}_sig{strSigma}_s{seed}"
    if (tcount_plug_simp != 30): return False #TEMP
    print("t",tcount_plug_simp,"\t",name) #TEMP
    os.makedirs(path, exist_ok=True)

    # save qasm file
    filename = os.path.join(path, name+".qasm")
    with open(filename, "w") as f:
        f.write(qasm)

    # save json file
    data = {
        "display_name": name,
        "file": f"{name}.qasm",
        "circuit_class": class_name,
        "benchmark_set": class_name,
        "provenance": {
            "generator": "pyzx.generate.cliffordT",
            "version": zx.__version__,
            "parameters": {
                "qubits": q,
                "depth": d,
                "seed": seed,
                "sigma": str(sigma)
            },
            "description": f"pyzx[{zx.__version__}].generate.cliffordT(qubits={q}, depth={d}, seed={seed}, sigma={sigma})"
        },
        "metrics": {
            "qubits": q,
            "depth": circ_depth,
            "sigma": strSigma,
            "t_count": tcount_raw,
            "pyzx_reduced_t_count": tcount_plug_simp,
            "gate_count": gate_count,
            "cnot_count": gate_counts["CNOT"],
            "had_count": gate_counts["HAD"]
        },
        "amplitude": {
            "real": amplitude.real if comp_amp else "TBC",
            "imag": amplitude.imag if comp_amp else "TBC"
        }
    }
    filename = os.path.join(path, name+".json")
    with open(filename, "w") as f:
        json.dump(data, f, indent=4)
    
    return True

In [ ]:
%%time
i = 0
q = 20
for d in range(1,80,1):
    i+=1
    print("             > ",i)
    gen_fixed_q(q,d*10,None,1,False)

             >  1
             >  2
             >  3
             >  4
             >  5
             >  6
             >  7
             >  8
             >  9
             >  10
             >  11
             >  12
             >  13
             >  14
             >  15
             >  16
             >  17
t 3 	 rfq_q20_d21_sigInf_s1
             >  18
t 3 	 rfq_q20_d25_sigInf_s1
             >  19
t 3 	 rfq_q20_d27_sigInf_s1
             >  20
t 4 	 rfq_q20_d29_sigInf_s1
             >  21
t 4 	 rfq_q20_d30_sigInf_s1
             >  22
t 8 	 rfq_q20_d31_sigInf_s1
             >  23
t 3 	 rfq_q20_d31_sigInf_s1
             >  24
             >  25
t 10 	 rfq_q20_d31_sigInf_s1
             >  26
t 14 	 rfq_q20_d34_sigInf_s1
             >  27
t 13 	 rfq_q20_d34_sigInf_s1
             >  28
t 14 	 rfq_q20_d34_sigInf_s1
             >  29
t 15 	 rfq_q20_d34_sigInf_s1
             >  30
t 17 	 rfq_q20_d35_sigInf_s1
             >  31
t 18 	 rfq_q20_d36_sigInf_s1
             >  32
t 

In [ ]:
%%time
i = 0
for q in range(2,42,2):
    for d in range(20,120,20):
        i+=1
        print("             > ",i)
        gen_fixed_t(q,d*10,None,1,False)

             >  1
             >  2
t 30 	 rct_q2_d289_sigInf_s1
             >  3
             >  4
             >  5
             >  6
             >  7
             >  8
             >  9
             >  10
             >  11
             >  12
             >  13
             >  14
             >  15
             >  16
             >  17
             >  18
             >  19
             >  20
             >  21
             >  22
             >  23
             >  24
             >  25
             >  26
             >  27
             >  28
             >  29
             >  30
             >  31
             >  32
             >  33
             >  34
             >  35
             >  36
             >  37
             >  38
             >  39
